# AquaTrack: test the SwimXYZ ViTPose-L model on real swim videos

This notebook **only runs** an existing model on your videos. Nothing is trained and no weights are changed.

- **Model:** ViTPose-L fine-tuned on SwimXYZ, 5 camera views ([willalexm/vitpose-l-swimxyz](https://huggingface.co/willalexm/vitpose-l-swimxyz)),
  checkpoint `vitpose-l-swimxyz-mv.pth`, 17 COCO body points.
- **Code it is built on:** [easy_ViTPose](https://github.com/JunkyByte/easy_ViTPose) (the model card says to load it with this repo's `ViTPose` class).
- **How it works:** ViTPose only looks *inside a box* around one person. So first a detector (or you) draws a box around the
  swimmer, then ViTPose finds the 17 joints inside that box.

**How to run**
1. `Runtime > Change runtime type > T4 GPU` (any GPU works).
2. Run the cells from top to bottom (`Shift+Enter`). Each section starts with a short explanation.
3. Settings you may want to change are in CAPITAL letters at the top of each cell.
4. Everything is saved to a **new** folder (`MyDrive/aquatrack_vitpose_swimxyz/<date_time>/`), so your RTMW-X and earlier ViTPose
   results are never touched.

**Sections:** 1 Environment Setup · 2 Model Download · 3 Video Upload · 4 Person Detection · 5 Pose Estimation ·
6 Visualization · 7 Results Download · 8 Failure Analysis

> Confidence scores tell you how sure the model is, **not** how accurate it is. To measure accuracy you need hand-labeled joints.

## 1. Environment Setup

Checks the GPU, installs the packages, and downloads the easy_ViTPose code at a fixed version (so it can't change under us).
We do **not** install easy_ViTPose's `requirements.txt`: it pins old versions that would break Colab's own packages.

In [ ]:
# 1.1  Check the GPU
import subprocess, torch

print("PyTorch", torch.__version__)
if torch.cuda.is_available():
    DEVICE = "cuda"
    GPU_NAME = torch.cuda.get_device_name(0)
    print("GPU found:", GPU_NAME)
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                         capture_output=True, text=True).stdout)
else:
    DEVICE = "cpu"
    GPU_NAME = "none (CPU)"
    print("No GPU. Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")
    print("(The notebook still works on CPU, just much slower: about 1-2 frames per second.)")

In [ ]:
# 1.2  Install packages and get easy_ViTPose
EASY_VITPOSE_COMMIT = "bb9860359e55b099a507c8000e360d48a27cc36d"   # latest commit on 2026-11-02

import os, sys
!pip install -q ultralytics huggingface_hub ffmpeg==1.4   # ffmpeg==1.4 is a tiny package easy_ViTPose imports
if not os.path.isdir("/content/easy_ViTPose"):
    !git clone -q https://github.com/JunkyByte/easy_ViTPose.git /content/easy_ViTPose
!cd /content/easy_ViTPose && git fetch -q --depth 1 origin {EASY_VITPOSE_COMMIT} ; git -C /content/easy_ViTPose checkout -q {EASY_VITPOSE_COMMIT} && git -C /content/easy_ViTPose log -1 --format="easy_ViTPose at %h (%cd)"

# Same imports as the model card: the easy_ViTPose code folder goes on the Python path.
EVP_DIR = "/content/easy_ViTPose/easy_ViTPose"
if EVP_DIR not in sys.path:
    sys.path.insert(0, EVP_DIR)
from vit_models.model import ViTPose
from configs.train_configs.ViTPose_large_coco_256x192 import model as VITPOSE_L_CFG   # ViTPose-L, 256x192, 17 joints
from vit_utils.top_down_eval import keypoints_from_heatmaps
print("easy_ViTPose imported OK")

In [ ]:
# 1.3  Output folder (a new one every time, so nothing old is overwritten)
import datetime, os

SAVE_TO_DRIVE = True          # False = keep results only in this Colab session (/content)

RUN_NAME = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE_DIR = "/content/drive/MyDrive/aquatrack_vitpose_swimxyz"
    except Exception as e:
        print("Drive not available, saving in /content instead:", e)
        BASE_DIR = "/content/aquatrack_vitpose_swimxyz"
else:
    BASE_DIR = "/content/aquatrack_vitpose_swimxyz"

OUT_DIR = os.path.join(BASE_DIR, RUN_NAME)
os.makedirs(os.path.join(OUT_DIR, "sample_frames"), exist_ok=True)
MODEL_DIR = "/content/vitpose_swimxyz_models"     # downloaded weights live here, separate from other experiments
os.makedirs(MODEL_DIR, exist_ok=True)
print("Results will be saved in:", OUT_DIR)

## 2. Model Download

Downloads `vitpose-l-swimxyz-mv.pth` (about 1.2 GB) from Hugging Face and loads it into easy_ViTPose's ViTPose-L
(ViT-Large backbone, 256×192 input, 17 heatmaps of 64×48), exactly as the model card shows.

The cell checks that **every** weight in the file matches the model (strict loading). If even one name or shape is wrong it
stops with an error instead of quietly running a half-loaded model.

In [ ]:
# 2.1  Download the checkpoint
from huggingface_hub import hf_hub_download
import hashlib, os

HF_REPO = "willalexm/vitpose-l-swimxyz"
CKPT_NAME = "vitpose-l-swimxyz-mv.pth"

CKPT_PATH = hf_hub_download(repo_id=HF_REPO, filename=CKPT_NAME, local_dir=MODEL_DIR)
size_gb = os.path.getsize(CKPT_PATH) / 1e9
h = hashlib.sha256()
with open(CKPT_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
CKPT_SHA256 = h.hexdigest()
print(f"Downloaded {CKPT_PATH} ({size_gb:.2f} GB)")
print("sha256:", CKPT_SHA256)

In [ ]:
# 2.2  Build ViTPose-L and load the weights (no training: eval mode, no gradients)
import copy, pickle, torch

def load_checkpoint(path):
    # First try PyTorch's safe loader (it only accepts plain tensors and numbers).
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except pickle.UnpicklingError as e:
        print("Safe loading refused this file:", str(e).splitlines()[0])
        print("The model card loads it with weights_only=False. That can run code stored in the file,")
        print("so only do it for files you trust. Set ALLOW_UNSAFE_LOAD = True below and run again.")
        if not ALLOW_UNSAFE_LOAD:
            raise
        return torch.load(path, map_location="cpu", weights_only=False)

ALLOW_UNSAFE_LOAD = False

def build_vitpose_l(ckpt_path):
    net = ViTPose(copy.deepcopy(VITPOSE_L_CFG))
    ckpt = load_checkpoint(ckpt_path)
    state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
    result = net.load_state_dict(state, strict=True)          # raises an error if anything does not match
    print("All", len(state), "weight tensors loaded (missing:", len(result.missing_keys),
          ", unexpected:", len(result.unexpected_keys), ")")
    net.eval().requires_grad_(False)
    return net, ckpt

model, ckpt = build_vitpose_l(CKPT_PATH)
model.to(DEVICE)
if isinstance(ckpt, dict):
    for k in ["epoch", "val_pck", "per_view_pck"]:
        if k in ckpt:
            print(f"  checkpoint {k}: {ckpt[k]}")
del ckpt

n_params = sum(p.numel() for p in model.parameters()) / 1e6
with torch.inference_mode():
    out = model(torch.zeros(1, 3, 256, 192, device=DEVICE))
print(f"ViTPose-L: {n_params:.0f} M parameters, output shape {tuple(out.shape)} (expected (1, 17, 64, 48))")
assert tuple(out.shape) == (1, 17, 64, 48), "Unexpected output shape: wrong model or config"
MODEL_LABEL = "ViTPose-L SwimXYZ Multi-View"

## 3. Video Upload

Pick **one** way to give the video (MP4 or MOV):
- `VIDEO_SOURCE = "upload"`: a file picker opens and you choose the video from your computer.
- `VIDEO_SOURCE = "drive"`: set `DRIVE_VIDEO_PATH` to a file in your Google Drive (Drive must be mounted, cell 1.3).

We first test a **short piece** (`TEST_START_SEC`, `TEST_SECONDS`). When it looks right, set `RUN_FULL_VIDEO = True`
and run sections 4 to 7 again.

In [ ]:
# 3.1  Choose the video
import cv2, os, shutil

VIDEO_SOURCE = "upload"      # "upload" or "drive"
DRIVE_VIDEO_PATH = "/content/drive/MyDrive/aquatrack_videos/my_swim.mp4"

if VIDEO_SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    if not up:
        raise SystemExit("No file chosen.")
    VIDEO_PATH = os.path.join("/content", list(up.keys())[0])
else:
    VIDEO_PATH = DRIVE_VIDEO_PATH
assert os.path.isfile(VIDEO_PATH), f"File not found: {VIDEO_PATH}"
assert VIDEO_PATH.lower().endswith((".mp4", ".mov", ".m4v")), "Please use an MP4 or MOV file"
print("Video:", VIDEO_PATH)

In [ ]:
# 3.2  Video facts, and which part we will process
import cv2, numpy as np
import matplotlib.pyplot as plt

TEST_START_SEC = 0.0     # where the test piece starts (seconds)
TEST_SECONDS = 8.0       # test piece length: 5-10 s is a good check
RUN_FULL_VIDEO = False   # True = process the whole video

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), "OpenCV could not open the video"
VID_W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); VID_H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
VID_FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0
VID_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
ok, first = cap.read()
cap.release()
assert ok, "Could not read the first frame"
if first.shape[:2] != (VID_H, VID_W):          # phone videos: OpenCV applies the rotation flag
    VID_H, VID_W = first.shape[:2]
print(f"Resolution: {VID_W} x {VID_H}")
print(f"FPS:        {VID_FPS:.2f}")
print(f"Frames:     {VID_FRAMES}")
print(f"Duration:   {VID_FRAMES / VID_FPS:.1f} s")

if RUN_FULL_VIDEO:
    START_FRAME, END_FRAME = 0, VID_FRAMES
else:
    START_FRAME = int(round(TEST_START_SEC * VID_FPS))
    END_FRAME = min(VID_FRAMES, START_FRAME + int(round(TEST_SECONDS * VID_FPS)))
print(f"Will process frames {START_FRAME} to {END_FRAME - 1} ({(END_FRAME - START_FRAME) / VID_FPS:.1f} s)")

def read_frame(idx):
    # One frame (BGR, as OpenCV gives it).
    c = cv2.VideoCapture(VIDEO_PATH)
    c.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, f = c.read()
    c.release()
    return f if ok else None

plt.figure(figsize=(10, 6)); plt.imshow(cv2.cvtColor(read_frame(START_FRAME), cv2.COLOR_BGR2RGB))
plt.title(f"Frame {START_FRAME} (pixel grid helps with manual boxes)"); plt.grid(alpha=0.4); plt.show()

## 4. Person Detection

ViTPose needs a box around the swimmer. There are two ways to get it:

- **Mode A (automatic):** YOLO finds the person. Submerged swimmers are hard for normal person detectors, so we use a
  low threshold. You can also use our own swim YOLO model from Drive (`DETECTOR_WEIGHTS`).
- **Mode B (manual):** you draw the box yourself on a few *key frames*. Between key frames the box slides smoothly
  (linear interpolation). With `FOLLOW_POSE = True` the box also follows the joints the model found in the previous frame,
  so a moving swimmer stays inside it.
- **Mode AB:** YOLO first; when it finds nobody, the manual/follow box is used. Every frame records **where its box came from**,
  so later we can tell detector failures from pose failures.

Only one swimmer is followed: the box closest to the previous one (or, at the start, the biggest confident box).

In [ ]:
# 4.1  Detector settings and helper functions
import numpy as np
from ultralytics import YOLO

DETECTION_MODE = "AB"          # "A" = YOLO only, "B" = manual only, "AB" = YOLO, manual/follow when YOLO misses
DETECTOR_WEIGHTS = "yolo11x.pt"   # downloads automatically. Or our swim model, e.g.
                                  # "/content/drive/MyDrive/aquatrack_swim_full/runs/swim_full/weights/best.pt"
DET_CONF = 0.15                # low on purpose: submerged swimmers get low scores
DET_IMGSZ = 1280               # bigger = finds small/far swimmers better, but slower
FOLLOW_POSE = True             # mode B/AB: move the box with the joints found in the previous frame
BOX_PAD = 0.15                 # extra space around the box on each side (15 %), so hands/feet are not cut off

detector = YOLO(DETECTOR_WEIGHTS)
print("Detector:", DETECTOR_WEIGHTS, "| classes:", detector.names if len(detector.names) < 5 else "COCO 80 classes")

def iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def yolo_people(frame_bgr):
    # All person boxes YOLO finds: list of (x1, y1, x2, y2, conf).
    r = detector(frame_bgr, conf=DET_CONF, imgsz=DET_IMGSZ, classes=[0], verbose=False,
                 device=0 if DEVICE == "cuda" else "cpu")[0]
    return [tuple(map(float, b)) for b in r.boxes.data[:, :5].cpu().numpy()]

def pick_swimmer(boxes, prev_box):
    # Keep one swimmer: overlapping the previous box, otherwise the biggest confident box.
    if not boxes:
        return None
    if prev_box is not None:
        best = max(boxes, key=lambda b: iou(b[:4], prev_box))
        if iou(best[:4], prev_box) > 0.1:
            return best
        pc = np.array([(prev_box[0] + prev_box[2]) / 2, (prev_box[1] + prev_box[3]) / 2])
        return min(boxes, key=lambda b: np.hypot((b[0] + b[2]) / 2 - pc[0], (b[1] + b[3]) / 2 - pc[1]))
    return max(boxes, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]) * b[4])

# Manual key-frame boxes: {frame_number: (x1, y1, x2, y2)}. Fill with cell 4.2 or type numbers here.
MANUAL_BOXES = {}

def manual_box(idx):
    # Box for frame idx from the key frames (straight-line blend between the two nearest ones).
    if not MANUAL_BOXES:
        return None
    keys = sorted(MANUAL_BOXES)
    if idx <= keys[0]:
        return tuple(MANUAL_BOXES[keys[0]])
    if idx >= keys[-1]:
        return tuple(MANUAL_BOXES[keys[-1]])
    for a, b in zip(keys, keys[1:]):
        if a <= idx <= b:
            t = (idx - a) / (b - a)
            return tuple((1 - t) * np.array(MANUAL_BOXES[a]) + t * np.array(MANUAL_BOXES[b]))

def box_from_pose(kpts, thr=0.3, min_points=5):
    # Box around the joints the model was fairly sure about in the previous frame.
    good = kpts[kpts[:, 2] >= thr]
    if len(good) < min_points:
        return None
    x1, y1 = good[:, 0].min(), good[:, 1].min()
    x2, y2 = good[:, 0].max(), good[:, 1].max()
    w, h = x2 - x1, y2 - y1
    return (x1 - 0.1 * w, y1 - 0.1 * h, x2 + 0.1 * w, y2 + 0.1 * h)

In [ ]:
# 4.2  (Mode B / AB) Draw a box on a key frame
# Run this cell once per key frame. Change KEY_FRAME each time (e.g. start, middle, end of the test piece).
# Drag a rectangle around the WHOLE swimmer (include the parts under water), then click "Save box".
# If the drawing tool does not work, type the box into MANUAL_BOXES in the cell above instead.
import base64, cv2
from IPython.display import display, Javascript

KEY_FRAME = START_FRAME
DRAW_NOW = DETECTION_MODE in ("B", "AB")   # set False to skip drawing (mode AB then only follows the joints)

def draw_box_on_frame(frame_bgr, max_w=960):
    from google.colab.output import eval_js
    h, w = frame_bgr.shape[:2]
    s = min(1.0, max_w / w)
    small = cv2.resize(frame_bgr, (int(w * s), int(h * s)))
    b64 = base64.b64encode(cv2.imencode(".jpg", small)[1]).decode()
    display(Javascript('''
    window.drawBox = function(b64) {
      return new Promise((resolve) => {
        const wrap = document.createElement('div');
        const img = new Image();
        const cv = document.createElement('canvas');
        const btn = document.createElement('button');
        btn.textContent = 'Save box'; btn.style.margin = '6px';
        const info = document.createElement('span');
        wrap.appendChild(cv); wrap.appendChild(document.createElement('br'));
        wrap.appendChild(btn); wrap.appendChild(info);
        document.body.appendChild(wrap);
        let box = null, start = null;
        img.onload = () => { cv.width = img.width; cv.height = img.height; draw(); };
        img.src = 'data:image/jpeg;base64,' + b64;
        const ctx = cv.getContext('2d');
        function draw() {
          ctx.drawImage(img, 0, 0);
          if (box) { ctx.strokeStyle = '#00ff00'; ctx.lineWidth = 2;
                     ctx.strokeRect(box[0], box[1], box[2] - box[0], box[3] - box[1]); }
        }
        function pos(e) { const r = cv.getBoundingClientRect();
          return [(e.clientX - r.left) * cv.width / r.width, (e.clientY - r.top) * cv.height / r.height]; }
        cv.onmousedown = (e) => { start = pos(e); };
        cv.onmousemove = (e) => { if (!start) return; const p = pos(e);
          box = [Math.min(start[0], p[0]), Math.min(start[1], p[1]), Math.max(start[0], p[0]), Math.max(start[1], p[1])];
          info.textContent = ' box: ' + box.map(Math.round).join(', '); draw(); };
        cv.onmouseup = () => { start = null; };
        btn.onclick = () => { if (!box) { info.textContent = ' draw a box first'; return; }
          wrap.remove(); resolve(box); };
      });
    };'''))
    box = eval_js(f'drawBox("{b64}")')
    return tuple(float(v) / s for v in box)

if DRAW_NOW:
    frame = read_frame(KEY_FRAME)
    MANUAL_BOXES[KEY_FRAME] = draw_box_on_frame(frame)
    print(f"Key frame {KEY_FRAME}: box", tuple(round(v) for v in MANUAL_BOXES[KEY_FRAME]))
print("All key frames so far:", {k: tuple(round(v) for v in b) for k, b in sorted(MANUAL_BOXES.items())})

In [ ]:
# 4.3  Preview: what YOLO finds on 6 frames of the test piece (before running the pose model)
import matplotlib.pyplot as plt

preview_idx = np.linspace(START_FRAME, END_FRAME - 1, 6).astype(int)
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, idx in zip(axes.ravel(), preview_idx):
    f = read_frame(idx)
    boxes = yolo_people(f)
    vis = f.copy()
    for b in boxes:
        cv2.rectangle(vis, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 255, 255), 3)
        cv2.putText(vis, f"{b[4]:.2f}", (int(b[0]), max(30, int(b[1]) - 8)), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
    mb = manual_box(idx)
    if mb is not None:
        cv2.rectangle(vis, (int(mb[0]), int(mb[1])), (int(mb[2]), int(mb[3])), (0, 255, 0), 3)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); ax.axis("off")
    ax.set_title(f"frame {idx}: YOLO found {len(boxes)} person box(es)")
plt.suptitle("yellow = YOLO person boxes with score, green = manual box"); plt.tight_layout(); plt.show()

## 5. Pose Estimation

For every frame:
1. Get the swimmer's box (section 4) and add `BOX_PAD` margin.
2. Widen the box to the 3:4 shape the model expects (192 wide × 256 high) and cut it out. Parts outside the video are black,
   like in easy_ViTPose.
3. Convert BGR → **RGB**, scale to 0-1, apply **ImageNet** normalization (mean 0.485/0.456/0.406, std 0.229/0.224/0.225).
4. ViTPose-L outputs 17 heatmaps (64×48). easy_ViTPose's own decoder (`keypoints_from_heatmaps`, unbiased + UDP, the same
   settings easy_ViTPose uses) turns each heatmap peak into an (x, y, confidence) point **in original video pixels**.

Joint order is COCO-17 (`0 nose … 15 left ankle, 16 right ankle`). "Left" means the **swimmer's** left.
`FLIP_TEST = True` also runs a mirrored copy and averages the two (standard ViTPose test trick, a bit more stable, 2× slower).

In [ ]:
# 5.1  Pose functions
import time, torch, cv2, numpy as np

KEYPOINT_NAMES = ["nose", "left_eye", "right_eye", "left_ear", "right_ear",
                  "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
                  "left_wrist", "right_wrist", "left_hip", "right_hip",
                  "left_knee", "right_knee", "left_ankle", "right_ankle"]
FLIP_PAIRS = [(1, 2), (3, 4), (5, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16)]
INPUT_W, INPUT_H = 192, 256                     # ViTPose-L input (width, height)
HEATMAP_W, HEATMAP_H = 48, 64
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
FLIP_TEST = False

def pad_box(box, img_w, img_h, pad=BOX_PAD):
    x1, y1, x2, y2 = box[:4]
    w, h = x2 - x1, y2 - y1
    x1, y1, x2, y2 = x1 - pad * w, y1 - pad * h, x2 + pad * w, y2 + pad * h
    # keep it inside the video (but allow the 3:4 widening below to go outside)
    return max(0, x1), max(0, y1), min(img_w, x2), min(img_h, y2)

def box_to_center_scale(box):
    # 3:4 (w:h) box around the same center, as big as needed to contain the box.
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    w, h = max(x2 - x1, 1.0), max(y2 - y1, 1.0)
    aspect = INPUT_W / INPUT_H
    if w / h < aspect:
        w = h * aspect
    else:
        h = w / aspect
    return np.array([cx, cy], np.float32), np.array([w, h], np.float32)

def crop_for_model(frame_bgr, center, scale):
    # Cut the 3:4 box out and resize to 192x256. Matrix follows the UDP convention:
    # box left edge -> pixel 0, box right edge -> pixel 191 (same for height).
    sx, sy = (INPUT_W - 1) / scale[0], (INPUT_H - 1) / scale[1]
    x0, y0 = center[0] - scale[0] / 2, center[1] - scale[1] / 2
    M = np.array([[sx, 0, -x0 * sx], [0, sy, -y0 * sy]], np.float32)
    return cv2.warpAffine(frame_bgr, M, (INPUT_W, INPUT_H), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

def to_tensor(crop_bgr):
    rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    x = (rgb - MEAN) / STD
    return torch.from_numpy(x.transpose(2, 0, 1)[None].copy())

def flip_back(heatmaps):
    # Undo a left-right mirror on heatmaps: flip horizontally and swap left/right channels.
    hm = heatmaps[..., ::-1].copy()
    for a, b in FLIP_PAIRS:
        hm[:, [a, b]] = hm[:, [b, a]]
    return hm

@torch.inference_mode()
def run_heatmaps(net, crop_bgr, flip_test=False):
    x = to_tensor(crop_bgr).to(DEVICE)
    hm = net(x).float().cpu().numpy()
    if flip_test:
        hm_f = net(torch.flip(x, dims=[3])).float().cpu().numpy()
        hm = (hm + flip_back(hm_f)) / 2
    return hm

def estimate_pose(net, frame_bgr, box, flip_test=None):
    # 17 x (x, y, confidence) in original video pixels, plus the crop the model saw.
    flip_test = FLIP_TEST if flip_test is None else flip_test
    h, w = frame_bgr.shape[:2]
    center, scale = box_to_center_scale(pad_box(box, w, h))
    crop = crop_for_model(frame_bgr, center, scale)
    hm = run_heatmaps(net, crop, flip_test)
    pts, prob = keypoints_from_heatmaps(hm, center=center[None], scale=scale[None],
                                        unbiased=True, use_udp=True)
    kpts = np.concatenate([pts[0], prob[0]], axis=1).astype(np.float32)   # (17, 3): x, y, conf
    return kpts, crop, hm[0]

In [ ]:
# 5.2  Run detection + pose on the chosen frames
from tqdm.auto import tqdm

KPT_THR = 0.3        # a joint counts as "found" above this confidence (only used for counting/drawing)

def run_video(net, start=None, end=None, mode=None):
    start = START_FRAME if start is None else start
    end = END_FRAME if end is None else end
    mode = DETECTION_MODE if mode is None else mode
    cap = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    results, prev_box, prev_kpts = [], None, None
    t_det = t_pose = 0.0
    for idx in tqdm(range(start, end), desc="frames"):
        ok, frame = cap.read()
        if not ok:
            break
        rec = {"frame": idx, "time_s": round(idx / VID_FPS, 3), "box": None, "box_source": "none",
               "det_conf": None, "n_person_boxes": 0, "keypoints": None, "mean_conf": None}
        box = None
        if mode in ("A", "AB"):
            t0 = time.perf_counter()
            boxes = yolo_people(frame)
            t_det += time.perf_counter() - t0
            rec["n_person_boxes"] = len(boxes)
            pick = pick_swimmer(boxes, prev_box)
            if pick is not None:
                box, rec["box_source"], rec["det_conf"] = pick[:4], "yolo", round(pick[4], 4)
        if box is None and mode in ("B", "AB"):
            follow = box_from_pose(prev_kpts) if (FOLLOW_POSE and prev_kpts is not None) else None
            if len(MANUAL_BOXES) >= 2:                 # several key frames: trust your boxes
                box, rec["box_source"] = manual_box(idx), "manual"
            elif follow is not None:                   # otherwise follow the swimmer's joints
                box, rec["box_source"] = follow, "follow_pose"
            elif manual_box(idx) is not None:          # one key frame: use it until the joints take over
                box, rec["box_source"] = manual_box(idx), "manual"
        if box is not None:
            t0 = time.perf_counter()
            kpts, _, _ = estimate_pose(net, frame, box)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            t_pose += time.perf_counter() - t0
            rec["box"] = [round(float(v), 1) for v in box[:4]]
            rec["keypoints"] = [[round(float(x), 2), round(float(y), 2), round(float(c), 4)] for x, y, c in kpts]
            rec["mean_conf"] = round(float(kpts[:, 2].mean()), 4)
            prev_box, prev_kpts = box[:4], kpts
        results.append(rec)
    cap.release()
    timing = {"detector_s": t_det, "pose_s": t_pose}
    return results, timing

t_start = time.perf_counter()
RESULTS, TIMING = run_video(model)
TIMING["total_s"] = time.perf_counter() - t_start
n = len(RESULTS); n_pose = sum(r["keypoints"] is not None for r in RESULTS)
print(f"Processed {n} frames in {TIMING['total_s']:.1f} s ({n / TIMING['total_s']:.1f} frames/s including video reading)")
print(f"Pose model alone: {1000 * TIMING['pose_s'] / max(n_pose, 1):.1f} ms per frame")
print(f"Frames with a swimmer box: {n_pose} / {n}")
from collections import Counter
print("Where the boxes came from:", dict(Counter(r["box_source"] for r in RESULTS)))

## 6. Visualization

Colors: **blue = swimmer's left side**, **orange = right side**, **green = head and center lines**.
Box colors: yellow = YOLO, green = manual, purple = follow-pose.
- `SHOW_LOW_CONF = False` hides joints below `KPT_THR`. `True` draws them as small grey hollow circles so you can see
  where the model *guessed*.
- `SHOW_BOX = True` draws the swimmer box.

Videos are written at the **original resolution**, then converted to H.264 MP4 so they play in normal players.
The side-by-side video is shrunk only if it would be wider than `COMPARISON_MAX_W`.

In [ ]:
# 6.1  Drawing settings and functions
SHOW_LOW_CONF = False
SHOW_BOX = True
COMPARISON_MAX_W = 3840

LEFT_COLOR, RIGHT_COLOR, MID_COLOR, LOW_COLOR = (255, 140, 30), (30, 140, 255), (60, 200, 60), (160, 160, 160)  # BGR
BONES = [  # (joint a, joint b, side)
    (5, 7, "L"), (7, 9, "L"), (11, 13, "L"), (13, 15, "L"), (5, 11, "L"),
    (6, 8, "R"), (8, 10, "R"), (12, 14, "R"), (14, 16, "R"), (6, 12, "R"),
    (5, 6, "M"), (11, 12, "M"), (0, 1, "L"), (1, 3, "L"), (0, 2, "R"), (2, 4, "R"),
]
SIDE_COLOR = {"L": LEFT_COLOR, "R": RIGHT_COLOR, "M": MID_COLOR}
BOX_COLOR = {"yolo": (0, 230, 255), "manual": (0, 255, 0), "follow_pose": (255, 0, 200)}

def joint_color(j):
    if j == 0:
        return MID_COLOR
    return LEFT_COLOR if j % 2 == 1 else RIGHT_COLOR

def draw_pose(frame_bgr, rec, thr=None, show_low=None, show_box=None):
    thr = KPT_THR if thr is None else thr
    show_low = SHOW_LOW_CONF if show_low is None else show_low
    show_box = SHOW_BOX if show_box is None else show_box
    img = frame_bgr.copy()
    s = max(1, round(max(img.shape[:2]) / 720))           # line/dot size grows with resolution
    if rec.get("keypoints") is None:
        cv2.putText(img, "no swimmer box", (20 * s, 40 * s), cv2.FONT_HERSHEY_SIMPLEX, 1.0 * s, (0, 0, 255), 2 * s)
        return img
    k = np.array(rec["keypoints"])
    if show_box and rec.get("box"):
        x1, y1, x2, y2 = map(int, rec["box"])
        col = BOX_COLOR.get(rec["box_source"], (255, 255, 255))
        cv2.rectangle(img, (x1, y1), (x2, y2), col, max(1, s))
        label = rec["box_source"] + (f" {rec['det_conf']:.2f}" if rec.get("det_conf") is not None else "")
        cv2.putText(img, label, (x1, max(20 * s, y1 - 6 * s)), cv2.FONT_HERSHEY_SIMPLEX, 0.6 * s, col, max(1, s))
    for a, b, side in BONES:
        if k[a, 2] >= thr and k[b, 2] >= thr:
            cv2.line(img, tuple(map(int, k[a, :2])), tuple(map(int, k[b, :2])), SIDE_COLOR[side], 3 * s, cv2.LINE_AA)
        elif show_low:
            cv2.line(img, tuple(map(int, k[a, :2])), tuple(map(int, k[b, :2])), LOW_COLOR, 1 * s, cv2.LINE_AA)
    for j, (x, y, c) in enumerate(k):
        if c >= thr:
            cv2.circle(img, (int(x), int(y)), 5 * s, joint_color(j), -1, cv2.LINE_AA)
            cv2.circle(img, (int(x), int(y)), 5 * s, (255, 255, 255), 1, cv2.LINE_AA)
        elif show_low:
            cv2.circle(img, (int(x), int(y)), 4 * s, LOW_COLOR, 1 * s, cv2.LINE_AA)
    cv2.putText(img, f"frame {rec['frame']}  mean conf {rec['mean_conf']:.2f}", (20 * s, 40 * s),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8 * s, (255, 255, 255), 2 * s)
    return img

def to_h264(src, dst):
    # Re-encode to H.264 + yuv420p (plays everywhere). Even width/height are required.
    r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-c:v", "libx264", "-preset", "veryfast",
                        "-crf", "20", "-pix_fmt", "yuv420p", "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2",
                        "-movflags", "+faststart", dst], capture_output=True, text=True)
    if r.returncode != 0:
        print("ffmpeg failed, keeping the OpenCV file instead:", r.stderr[-500:])
        shutil.move(src, dst)
    else:
        os.remove(src)

In [ ]:
# 6.2  Write vitpose_result.mp4 and vitpose_comparison.mp4
import subprocess, shutil, os

RESULT_MP4 = os.path.join(OUT_DIR, "vitpose_result.mp4")
COMPARE_MP4 = os.path.join(OUT_DIR, "vitpose_comparison.mp4")
cmp_scale = min(1.0, COMPARISON_MAX_W / (2 * VID_W))
cmp_size = (int(2 * VID_W * cmp_scale), int(VID_H * cmp_scale))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
w_res = cv2.VideoWriter("/content/_tmp_result.mp4", fourcc, VID_FPS, (VID_W, VID_H))
w_cmp = cv2.VideoWriter("/content/_tmp_compare.mp4", fourcc, VID_FPS, cmp_size)
by_frame = {r["frame"]: r for r in RESULTS}
cap = cv2.VideoCapture(VIDEO_PATH); cap.set(cv2.CAP_PROP_POS_FRAMES, RESULTS[0]["frame"])
for rec in tqdm(RESULTS, desc="drawing"):
    ok, frame = cap.read()
    if not ok:
        break
    over = draw_pose(frame, rec)
    w_res.write(over)
    side = np.hstack([frame, over])
    if cmp_scale < 1.0:
        side = cv2.resize(side, cmp_size, interpolation=cv2.INTER_AREA)
    w_cmp.write(side)
cap.release(); w_res.release(); w_cmp.release()
to_h264("/content/_tmp_result.mp4", RESULT_MP4)
to_h264("/content/_tmp_compare.mp4", COMPARE_MP4)
print("Saved", RESULT_MP4, f"({VID_W}x{VID_H})")
print("Saved", COMPARE_MP4, f"({cmp_size[0]}x{cmp_size[1]})")

In [ ]:
# 6.3  Watch the comparison video here (small preview)
from IPython.display import HTML
from base64 import b64encode

PREVIEW = "/content/_preview.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", COMPARE_MP4, "-vf", "scale=1280:-2",
                "-c:v", "libx264", "-crf", "28", "-pix_fmt", "yuv420p", PREVIEW])
if os.path.getsize(PREVIEW) < 60e6:
    data = b64encode(open(PREVIEW, "rb").read()).decode()
    display(HTML(f'<video width=1000 controls loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))
else:
    print("Video too long to preview here. Download it in section 7.")

In [ ]:
# 6.4  Sample frames: evenly spaced + the lowest-confidence frames
import json

N_EVEN, N_WORST = 8, 4
with_pose = [r for r in RESULTS if r["keypoints"] is not None]
even = [RESULTS[i] for i in np.linspace(0, len(RESULTS) - 1, N_EVEN).astype(int)]
worst = sorted(with_pose, key=lambda r: r["mean_conf"])[:N_WORST]
chosen = {r["frame"]: tag for tag, group in (("even", even), ("lowconf", worst)) for r in group}
SAMPLE_DIR = os.path.join(OUT_DIR, "sample_frames")
for fidx, tag in sorted(chosen.items()):
    img = draw_pose(read_frame(fidx), by_frame[fidx], show_low=True)
    cv2.imwrite(os.path.join(SAMPLE_DIR, f"frame_{fidx:06d}_{tag}.jpg"), img)

show = sorted(chosen.items())[:12]
cols = 4; rows = (len(show) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(20, 4.2 * rows))
for ax in np.atleast_1d(axes).ravel():
    ax.axis("off")
for ax, (fidx, tag) in zip(np.atleast_1d(axes).ravel(), show):
    ax.imshow(cv2.cvtColor(draw_pose(read_frame(fidx), by_frame[fidx], show_low=True), cv2.COLOR_BGR2RGB))
    mc = by_frame[fidx]["mean_conf"]
    ax.set_title(f"frame {fidx} ({tag})" + (f" mean conf {mc:.2f}" if mc is not None else " no box"))
plt.tight_layout(); plt.show()
print(f"Saved {len(chosen)} frames to {SAMPLE_DIR} (grey hollow circles = low-confidence joints)")

## 7. Results Download

Saves `vitpose_keypoints.json` and `inference_report.txt`, then zips the whole result folder and downloads it.
(If you saved to Drive, everything is also in `MyDrive/aquatrack_vitpose_swimxyz/<date_time>/`.)

**JSON format:** one entry per frame. `keypoints` is a list of 17 `[x, y, confidence]` in original video pixels,
in the order of `keypoint_names` (note: easy_ViTPose's own JSON uses `[y, x, score]`; ours is `[x, y, conf]`).
`box_source` tells where the box came from (`yolo`, `manual`, `follow_pose`, or `none`).

In [ ]:
# 7.1  Save keypoints JSON and the report
import json, statistics

def conf_stats(results):
    K = np.array([r["keypoints"] for r in results if r["keypoints"] is not None])   # (frames, 17, 3)
    if len(K) == 0:
        return None
    c = K[:, :, 2]
    return {
        "frames_with_pose": int(len(K)),
        "mean": float(c.mean()), "median": float(np.median(c)),
        "p10": float(np.percentile(c, 10)), "p90": float(np.percentile(c, 90)),
        "share_joints_above_thr": float((c >= KPT_THR).mean()),
        "per_joint_mean": {n: float(v) for n, v in zip(KEYPOINT_NAMES, c.mean(0))},
        "per_joint_share_above_thr": {n: float(v) for n, v in zip(KEYPOINT_NAMES, (c >= KPT_THR).mean(0))},
    }

STATS = conf_stats(RESULTS)
meta = {
    "model": MODEL_LABEL, "hf_repo": HF_REPO, "checkpoint": CKPT_NAME, "checkpoint_sha256": CKPT_SHA256,
    "code": f"easy_ViTPose @ {EASY_VITPOSE_COMMIT}", "input_size_wh": [INPUT_W, INPUT_H],
    "video": os.path.basename(VIDEO_PATH), "width": VID_W, "height": VID_H, "fps": VID_FPS,
    "frames_processed": [RESULTS[0]["frame"], RESULTS[-1]["frame"]],
    "detection_mode": DETECTION_MODE, "detector": DETECTOR_WEIGHTS, "det_conf": DET_CONF, "box_pad": BOX_PAD,
    "flip_test": FLIP_TEST, "keypoint_format": "[x, y, confidence] in original video pixels",
    "keypoint_names": KEYPOINT_NAMES, "skeleton": [[a, b] for a, b, _ in BONES],
    "manual_key_boxes": {str(k): [round(float(v), 1) for v in b] for k, b in MANUAL_BOXES.items()},
}
with open(os.path.join(OUT_DIR, "vitpose_keypoints.json"), "w") as f:
    json.dump({"meta": meta, "frames": RESULTS}, f)

n = len(RESULTS)
n_box = sum(r["box"] is not None for r in RESULTS)
src = Counter(r["box_source"] for r in RESULTS)
yolo_tried = DETECTION_MODE in ("A", "AB")
lines = [
    "AquaTrack - ViTPose-L SwimXYZ inference test",
    "=" * 50,
    f"Date:             {datetime.datetime.now():%Y-%m-%d %H:%M}",
    f"Model:            {MODEL_LABEL}",
    f"Checkpoint:       {HF_REPO} / {CKPT_NAME}",
    f"Checkpoint sha256:{CKPT_SHA256}",
    f"Code:             easy_ViTPose @ {EASY_VITPOSE_COMMIT[:7]} (ViTPose-L, input 256x192, 17 COCO keypoints)",
    f"GPU:              {GPU_NAME}",
    f"Video:            {os.path.basename(VIDEO_PATH)} ({VID_W}x{VID_H}, {VID_FPS:.2f} fps, {VID_FRAMES / VID_FPS:.1f} s)",
    f"Frames processed: {n} (frames {RESULTS[0]['frame']}-{RESULTS[-1]['frame']}, {n / VID_FPS:.1f} s)",
    f"Detection mode:   {DETECTION_MODE} (detector {DETECTOR_WEIGHTS}, conf >= {DET_CONF}, box padding {BOX_PAD:.0%})",
    f"Flip test:        {FLIP_TEST}",
    "",
    "Speed",
    f"  Total:          {TIMING['total_s']:.1f} s = {n / TIMING['total_s']:.1f} frames/s (reading + detection + pose)",
    f"  Detector:       {1000 * TIMING['detector_s'] / max(n, 1):.1f} ms/frame" if yolo_tried else "  Detector:       not used",
    f"  Pose (ViTPose): {1000 * TIMING['pose_s'] / max(n_box, 1):.1f} ms/frame",
    "",
    "Swimmer detection",
    f"  Frames with a swimmer box: {n_box} / {n} ({n_box / max(n, 1):.0%})",
    f"  Frames with no box (detection failed): {n - n_box}",
    f"  Box source: YOLO {src.get('yolo', 0)}, manual {src.get('manual', 0)}, follow-pose {src.get('follow_pose', 0)}, none {src.get('none', 0)}",
]
if yolo_tried:
    lines.append(f"  YOLO found the swimmer in {src.get('yolo', 0)} / {n} frames ({src.get('yolo', 0) / max(n, 1):.0%})")
lines += ["", "Keypoint confidence (model's own certainty, NOT accuracy)"]
if STATS:
    lines += [
        f"  Mean {STATS['mean']:.3f}, median {STATS['median']:.3f}, 10th pct {STATS['p10']:.3f}, 90th pct {STATS['p90']:.3f}",
        f"  Joints above {KPT_THR}: {STATS['share_joints_above_thr']:.0%}",
        "  Per joint (mean confidence / share of frames above threshold):",
    ]
    for name in KEYPOINT_NAMES:
        lines.append(f"    {name:15s} {STATS['per_joint_mean'][name]:.3f}   {STATS['per_joint_share_above_thr'][name]:.0%}")
else:
    lines.append("  No frames with a pose.")
lines += ["", "Note: confidence scores show how sure the model is, not whether the joints are in the right place.",
          "Measuring accuracy needs hand-labeled ground-truth joints (e.g. a few CVAT-labeled frames)."]
REPORT = "\n".join(lines)
with open(os.path.join(OUT_DIR, "inference_report.txt"), "w") as f:
    f.write(REPORT + "\n")
print(REPORT)

In [ ]:
# 7.2  Download everything as one zip
ZIP_PATH = shutil.make_archive(f"/content/vitpose_swimxyz_{RUN_NAME}", "zip", OUT_DIR)
print("Zip:", ZIP_PATH, f"({os.path.getsize(ZIP_PATH) / 1e6:.1f} MB)")
print("Folder:", OUT_DIR, "->", sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files
    files.download(ZIP_PATH)
except Exception as e:
    print("Automatic download not available here; use the folder icon on the left.", e)

## 8. Failure Analysis

Run this when the skeleton looks wrong. It separates four possible causes:

| Cause | How we check it |
|---|---|
| 1. YOLO did not find the swimmer | 8.1 timeline: frames with **no box** are detector failures |
| 2. ViTPose fails even with a correct box | 8.2: give a **hand-drawn box** on bad frames. Still bad → pose model problem |
| 3. Our preprocessing / coordinate conversion is wrong | 8.3: same code on a normal photo of people, with the original COCO ViTPose-L, plus a mirror test |
| 4. Underwater occlusion, reflection, refraction | 8.4: which joints are weak, above vs. below the waterline, extra "people" boxes (reflections) |

Nothing here trains or changes the model.

In [ ]:
# 8.1  Timeline: detector score and pose confidence for every frame
LOW_POSE = 0.35     # mean confidence below this = "pose looks unsure"

frames = np.array([r["frame"] for r in RESULTS])
det = np.array([r["det_conf"] if r["det_conf"] is not None else np.nan for r in RESULTS])
pose = np.array([r["mean_conf"] if r["mean_conf"] is not None else np.nan for r in RESULTS])
no_box = np.array([r["box"] is None for r in RESULTS])

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(frames, det, ".", ms=4, label="YOLO person score", color="goldenrod")
ax.plot(frames, pose, "-", lw=1.5, label="ViTPose mean joint confidence", color="tab:blue")
for f in frames[no_box]:
    ax.axvspan(f - 0.5, f + 0.5, color="red", alpha=0.15, lw=0)
ax.axhline(LOW_POSE, color="tab:blue", ls=":", lw=1)
ax.set_ylim(0, 1); ax.set_xlabel("frame"); ax.legend(loc="lower right")
ax.set_title("red bands = no swimmer box (detector failure)"); plt.show()

DETECTOR_FAIL = [int(f) for f in frames[no_box]]
POSE_UNSURE = [r["frame"] for r in RESULTS if r["mean_conf"] is not None and r["mean_conf"] < LOW_POSE]
print(f"Detector failures (no box):        {len(DETECTOR_FAIL)} frames  {DETECTOR_FAIL[:15]}{' ...' if len(DETECTOR_FAIL) > 15 else ''}")
print(f"Box found but pose unsure (<{LOW_POSE}): {len(POSE_UNSURE)} frames  {POSE_UNSURE[:15]}{' ...' if len(POSE_UNSURE) > 15 else ''}")
print(f"Frames with more than one person box (possible reflections/other swimmers): "
      f"{sum(r['n_person_boxes'] > 1 for r in RESULTS)}")

In [ ]:
# 8.2  Retry bad frames with a hand-drawn box
# For each frame you draw the box around the WHOLE swimmer. We then run ViTPose with your box and compare.
#  - your box gives a good skeleton           -> the problem was the detector's box (cause 1)
#  - still bad with a correct box             -> the pose model itself struggles here (cause 2 or 4)
N_RETRY = 3
USE_DRAWING_TOOL = True       # False: type boxes into TYPED_BOXES instead
TYPED_BOXES = {}              # e.g. {120: (400, 300, 1500, 700)}

retry_frames = (DETECTOR_FAIL[:: max(1, len(DETECTOR_FAIL) // N_RETRY)][:N_RETRY] +
                sorted(POSE_UNSURE, key=lambda f: by_frame[f]["mean_conf"])[:N_RETRY])
retry_frames = list(TYPED_BOXES) if (TYPED_BOXES and not USE_DRAWING_TOOL) else sorted(set(retry_frames))
retry_frames = [f for f in retry_frames if f in by_frame]      # only frames processed in section 5
print("Frames to retry:", retry_frames)

RETRY = []
for fidx in retry_frames:
    frame = read_frame(fidx)
    mbox = TYPED_BOXES.get(fidx) if fidx in TYPED_BOXES else (draw_box_on_frame(frame) if USE_DRAWING_TOOL else None)
    if mbox is None:
        continue
    kpts, crop, _ = estimate_pose(model, frame, mbox)
    rec_m = {"frame": fidx, "box": list(mbox), "box_source": "manual", "det_conf": None,
             "keypoints": kpts.tolist(), "mean_conf": float(kpts[:, 2].mean())}
    rec_a = by_frame[fidx]
    RETRY.append((fidx, rec_a, rec_m))
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    axes[0].imshow(cv2.cvtColor(draw_pose(frame, rec_a, show_low=True), cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"frame {fidx}: automatic ({rec_a['box_source']})" +
                      (f", mean conf {rec_a['mean_conf']:.2f}" if rec_a['mean_conf'] is not None else ", no box"))
    axes[1].imshow(cv2.cvtColor(draw_pose(frame, rec_m, show_low=True), cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"frame {fidx}: your box, mean conf {rec_m['mean_conf']:.2f}")
    for a in axes: a.axis("off")
    plt.tight_layout(); plt.show()

for fidx, a, m in RETRY:
    before = "no box" if a["mean_conf"] is None else f"{a['mean_conf']:.2f}"
    print(f"frame {fidx}: automatic {before} -> your box {m['mean_conf']:.2f}")
print("Look at the pictures, not only the numbers: a confident skeleton can still be in the wrong place.")

In [ ]:
# 8.3  Is our preprocessing / coordinate conversion correct?
# Test A: run the SAME code with the ORIGINAL COCO ViTPose-L (easy_ViTPose's own weights, trained on real people)
#         on a normal photo of standing people. If that skeleton is right, our crop, RGB, normalization and
#         heatmap-to-pixel conversion are right too.
# Test B: run each crop mirrored, mirror the answer back and swap left/right. A correct pipeline gives nearly the
#         same joints. Big differences point to left/right confusion.
# Test C: show the crop ViTPose actually sees, with its strongest heatmap, for one swim frame.
import urllib.request
from huggingface_hub import list_repo_files

# --- original COCO ViTPose-L (only for this check)
coco_files = [f for f in list_repo_files("JunkyByte/easy_ViTPose") if f.endswith("vitpose-l-coco.pth")]
print("COCO ViTPose-L file:", coco_files)
COCO_CKPT = hf_hub_download("JunkyByte/easy_ViTPose", coco_files[0], local_dir=MODEL_DIR)
coco_model, _ = build_vitpose_l(COCO_CKPT)
coco_model.to(DEVICE)

TEST_IMG_URL = "https://ultralytics.com/images/bus.jpg"
urllib.request.urlretrieve(TEST_IMG_URL, "/content/_people.jpg")
people = cv2.imread("/content/_people.jpg")
boxes = sorted(yolo_people(people), key=lambda b: -b[4])[:4]
fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for ax, (net, name) in zip(axes, [(coco_model, "original COCO ViTPose-L"), (model, "SwimXYZ ViTPose-L")]):
    vis = people.copy()
    for b in boxes:
        k, _, _ = estimate_pose(net, people, b)
        vis = draw_pose(vis, {"frame": 0, "box": list(b[:4]), "box_source": "yolo", "det_conf": b[4],
                              "keypoints": k.tolist(), "mean_conf": float(k[:, 2].mean())})
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); ax.set_title(name); ax.axis("off")
plt.suptitle("Test A: standing people. Left panel should look right; if it does, preprocessing + coordinates are OK")
plt.tight_layout(); plt.show()

# --- Test B: mirror test on swim frames that have a box
def mirror_test(net, frame, box):
    h, w = frame.shape[:2]
    k1, _, _ = estimate_pose(net, frame, box, flip_test=False)
    fl = frame[:, ::-1].copy()
    fbox = (w - box[2], box[1], w - box[0], box[3])
    k2, _, _ = estimate_pose(net, fl, fbox, flip_test=False)
    k2[:, 0] = (w - 1) - k2[:, 0]
    for a, b in FLIP_PAIRS:
        k2[[a, b]] = k2[[b, a]]
    torso = np.linalg.norm(k1[5, :2] - k1[12, :2]) + 1e-6       # shoulder-to-opposite-hip length as a size unit
    return np.linalg.norm(k1[:, :2] - k2[:, :2], axis=1) / torso

test_frames = [r for r in with_pose][:: max(1, len(with_pose) // 10)][:10]
d = np.array([mirror_test(model, read_frame(r["frame"]), r["box"]) for r in test_frames])
print("\nTest B: mirror test (difference / torso length; under ~0.1 is consistent)")
for j, name in enumerate(KEYPOINT_NAMES):
    flag = "  <- unstable / left-right confusion?" if np.median(d[:, j]) > 0.25 else ""
    print(f"  {name:15s} median {np.median(d[:, j]):.2f}{flag}")

# --- Test C: what the model sees
r0 = min(with_pose, key=lambda r: r["mean_conf"])
f0 = read_frame(r0["frame"])
k0, crop0, hm0 = estimate_pose(model, f0, r0["box"])
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
axes[0].imshow(cv2.cvtColor(crop0, cv2.COLOR_BGR2RGB)); axes[0].set_title(f"crop the model sees (frame {r0['frame']})")
axes[1].imshow(cv2.cvtColor(crop0, cv2.COLOR_BGR2RGB))
axes[1].imshow(cv2.resize(hm0.max(0), (INPUT_W, INPUT_H)), alpha=0.55, cmap="jet"); axes[1].set_title("all 17 heatmaps (max)")
axes[2].bar(range(17), k0[:, 2]); axes[2].set_xticks(range(17)); axes[2].set_xticklabels(KEYPOINT_NAMES, rotation=90)
axes[2].set_title("confidence per joint")
for a in axes[:2]: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 8.4  Underwater occlusion, reflection, refraction
# Side views: set WATERLINE_Y to the pixel row of the water surface (look at the grid in 3.2) to compare joints
# above vs. below it. Overhead views: leave it None; look at which joints are weak instead.
WATERLINE_Y = None

K = np.array([r["keypoints"] for r in with_pose])          # (frames, 17, 3)
c = K[:, :, 2]
order = np.argsort(c.mean(0))
print("Weakest joints (mean confidence):")
for j in order[:6]:
    print(f"  {KEYPOINT_NAMES[j]:15s} {c[:, j].mean():.2f}")
print("Strongest joints:")
for j in order[-4:][::-1]:
    print(f"  {KEYPOINT_NAMES[j]:15s} {c[:, j].mean():.2f}")

if WATERLINE_Y is not None:
    above = K[:, :, 1] < WATERLINE_Y
    print(f"\nAbove waterline: mean confidence {c[above].mean():.2f} ({above.mean():.0%} of joints)")
    print(f"Below waterline: mean confidence {c[~above].mean():.2f} ({(~above).mean():.0%} of joints)")
    print("Near the line (+/- 3% of height), refraction and splash are worst:",
          f"{c[np.abs(K[:, :, 1] - WATERLINE_Y) < 0.03 * VID_H].mean():.2f}")

# Reflections: extra person boxes that are mirror images (same x, other side of the swimmer) often come from the
# water surface. Show the frame with the most person boxes.
multi = max(RESULTS, key=lambda r: r["n_person_boxes"])
if multi["n_person_boxes"] > 1:
    f = read_frame(multi["frame"]); vis = f.copy()
    for b in yolo_people(f):
        cv2.rectangle(vis, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 255, 255), 3)
    plt.figure(figsize=(12, 6)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis("off")
    plt.title(f"frame {multi['frame']}: {multi['n_person_boxes']} person boxes (reflection or other swimmers?)"); plt.show()

# Wrist/ankle jumps: big frame-to-frame jumps of a confident joint often mean the model switched to a reflection
# or the other limb.
jumps = []
for j in [9, 10, 15, 16]:
    xy, cc = K[:, j, :2], c[:, j]
    size = np.linalg.norm(K[:, 5, :2] - K[:, 12, :2], axis=1) + 1e-6
    step = np.linalg.norm(np.diff(xy, axis=0), axis=1) / size[1:]
    ok = (cc[1:] > KPT_THR) & (cc[:-1] > KPT_THR)
    jumps.append((KEYPOINT_NAMES[j], int(((step > 0.5) & ok).sum())))
print("\nSudden jumps (> half a torso length between frames, both frames confident):", jumps)

In [ ]:
# 8.5  Summary: most likely cause (rule of thumb, check against the pictures above)
n = len(RESULTS)
det_fail_rate = len(DETECTOR_FAIL) / max(n, 1)
unsure_rate = len(POSE_UNSURE) / max(n, 1)
improved = [m["mean_conf"] - (a["mean_conf"] or 0) for _, a, m in RETRY] if RETRY else []
summary = [f"Detector failed on {det_fail_rate:.0%} of frames; pose unsure on {unsure_rate:.0%} of frames with a box."]
if det_fail_rate > 0.2:
    summary.append("-> Detection is a main problem (cause 1). Try DETECTION_MODE='AB' with key-frame boxes, a lower DET_CONF, "
                   "DET_IMGSZ=1280, or our swim YOLO as DETECTOR_WEIGHTS.")
if improved:
    summary.append(f"Hand-drawn boxes changed mean confidence by {np.mean(improved):+.2f} on {len(improved)} frames.")
    if np.mean(improved) > 0.1:
        summary.append("-> Better boxes help a lot: the automatic box was part of the problem (cause 1).")
    else:
        summary.append("-> Even good boxes don't help much: the pose model itself struggles on these frames (cause 2/4).")
summary.append("Cause 3 (preprocessing) is ruled out only if Test A's left panel looked right and Test B was mostly < 0.1.")
summary.append("If wrists/ankles are the weakest joints or below-waterline joints score much lower, underwater "
               "occlusion/refraction (cause 4) is the likely reason.")
print("\n".join(summary))
with open(os.path.join(OUT_DIR, "inference_report.txt"), "a") as f:
    f.write("\nFailure analysis\n" + "\n".join(summary) + "\n")
print("\n(Added to inference_report.txt. Run 7.2 again to download the updated zip.)")